<a href="https://colab.research.google.com/github/deepakawl/supplychain-analytics-teaching/blob/main/Network_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Network Analysis

In [ ]:
!pip install networkx matplotlib pandas
import pandas as pd, networkx as nx
Import matplotlib.pyplot as plt

In [ ]:
nodes = pd.read_csv('SunOilNodes.csv')
edges = pd.read_csv('SunOilEdges.csv')


In [ ]:
G = nx.from_pandas_edgelist(
    edges, source='Source', target='Target',
    edge_attr=['Cost'], create_using=nx.DiGraph()
)

attrs = nodes.set_index('ID').to_dict('index')
nx.set_node_attributes(G, attrs)

print(f'Nodes: {G.number_of_nodes()} | Edges: {G.number_of_edges()}')

## Shortest Path

In [ ]:
path = nx.dijkstra_path(G, 'Asia', 'N. America', weight='Cost')
cost = nx.dijkstra_path_length(G, 'Asia', 'N. America', weight='Cost')

print(f'Route: {path}  |  Cost: {cost}')  # → ['Asia', 'Asia'] cost=115 (direct)
# All-pairs shortest path costs

apsp = dict(nx.all_pairs_dijkstra_path_length(G, weight='Cost'))

## Centrality Measures

In [1]:
deg  = nx.degree_centrality(G)                         # degree
btw  = nx.betweenness_centrality(G, weight='Cost')     # betweenness
cls  = nx.closeness_centrality(G, distance='Cost')     # closeness

# Print as sorted DataFrame
pd.DataFrame({'Degree':deg,'Betweenness':btw,'Closeness':cls}).sort_values('Betweenness',ascending=False)

NameError: name 'nx' is not defined

## Maxflow

In [ ]:
# Capacity-weighted graph (use LowCap as capacity attribute)
# Add capacity to edges from nodes_df first, then:
flow_val, flow_dict = nx.maximum_flow(G, 'Asia', 'N. America', capacity='capacity')
print(f'Max Flow Asia → N.America: {flow_val}')
# nx.min_cost_flow(G)  ← requires 'demand','capacity','weight' on all nodes/edges

## Visualize SC Network

In [ ]:
pos = nx.circular_layout(G)  # or spring_layout, shell_layout

# Node size proportional to Demand
sizes = [nodes_df.set_index('ID').loc[n,'Demand']*120 for n in G.nodes()]

# Edge width: thicker = cheaper route (1/cost)
widths = [300 / G[u][v]['Cost'] for u,v in G.edges()]
fig, ax = plt.subplots(figsize=(8,6))

In [ ]:
nx.draw_networkx_nodes(G, pos, node_size=sizes,
                       node_color='steelblue', alpha=0.9, ax=ax)

nx.draw_networkx_labels(G, pos, font_size=10,
                        font_color='white', ax=ax)

nx.draw_networkx_edges(G, pos, width=widths,
                       edge_color='#5D6D7E', arrows=True,
                       arrowsize=18, ax=ax)

edge_labels = nx.get_edge_attributes(G,'Cost')

nx.draw_networkx_edge_labels(G, pos,
                             edge_labels=edge_labels, font_size=7)

plt.title('SunOil SC Network (node size = demand)')
plt.axis('off'); plt.tight_layout(); plt.show()